<a href="https://colab.research.google.com/github/pngl2415/2.6-openMP/blob/main/Actividad_3_2_CUDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Mon Nov  2 08:20:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [26]:
%%writefile operaciones_arreglos.cu
#include <cstdio>
#include <new>
#include <chrono>
#include <cuda_runtime.h>

#define THREADS_PER_BLOCK 256
#define MODO_MASIVO 1



using Reloj = std::chrono::steady_clock;

// ==========================================
// EQUIPO 01 - CONFIGURACION
// ==========================================





// PRUEBA MASIVA



constexpr int N = 4000000;
constexpr int HILOS = 256;
constexpr bool DETALLADO = false;



double milisegundos(Reloj::time_point inicio,
                    Reloj::time_point fin) {
    return std::chrono::duration<double, std::milli>(
        fin - inicio).count();
}

// ==========================================
// KERNEL 1: INICIALIZAR
// ==========================================

__global__ void inicializarArreglo(
    int* arreglo, int n, bool mostrar) {

    int globalIdx =
        blockIdx.x * blockDim.x + threadIdx.x;

    if (globalIdx < n) {
        arreglo[globalIdx] = (globalIdx % 40) + 1;

        if (mostrar) {
            printf("[Equipo: 01] [DEVICE] "
                   "[Bloque CUDA: %d] [Hilo CUDA: %d] "
                   "[Indice Global: %d] "
                   "[Operacion: Inicializar] "
                   "-> Valor: %d\n",
                   blockIdx.x, threadIdx.x,
                   globalIdx, arreglo[globalIdx]);
        }
    }
}

// ==========================================
// KERNEL 2: SUMA
// ==========================================

__global__ void sumarArreglos(
    const int* A, const int* B,
    int* C, int n, bool mostrar) {

    int globalIdx =
        blockIdx.x * blockDim.x + threadIdx.x;

    if (globalIdx < n) {
        C[globalIdx] = A[globalIdx] + B[globalIdx];

        if (mostrar) {
            printf("[Equipo: 01] [DEVICE] "
                   "[Bloque CUDA: %d] [Hilo CUDA: %d] "
                   "[Indice Global: %d] "
                   "[Operacion: Suma] %d + %d "
                   "-> Resultado: %d\n",
                   blockIdx.x, threadIdx.x, globalIdx,
                   A[globalIdx], B[globalIdx],
                   C[globalIdx]);
        }
    }
}

// ==========================================
// KERNEL 3: RESTA
// ==========================================

__global__ void restarArreglos(
    const int* A, const int* B,
    int* C, int n, bool mostrar) {

    int globalIdx =
        blockIdx.x * blockDim.x + threadIdx.x;

    if (globalIdx < n) {
        C[globalIdx] = A[globalIdx] - B[globalIdx];

        if (mostrar) {
            printf("[Equipo: 01] [DEVICE] "
                   "[Bloque CUDA: %d] [Hilo CUDA: %d] "
                   "[Indice Global: %d] "
                   "[Operacion: Resta] %d - %d "
                   "-> Resultado: %d\n",
                   blockIdx.x, threadIdx.x, globalIdx,
                   A[globalIdx], B[globalIdx],
                   C[globalIdx]);
        }
    }
}

// ==========================================
// KERNEL 4: MULTIPLICACION
// ==========================================

__global__ void multiplicarArreglos(
    const int* A, const int* B,
    int* C, int n, bool mostrar) {

    int globalIdx =
        blockIdx.x * blockDim.x + threadIdx.x;

    if (globalIdx < n) {
        C[globalIdx] = A[globalIdx] * B[globalIdx];

        if (mostrar) {
            printf("[Equipo: 01] [DEVICE] "
                   "[Bloque CUDA: %d] [Hilo CUDA: %d] "
                   "[Indice Global: %d] "
                   "[Operacion: Multiplicacion] %d * %d "
                   "-> Resultado: %d\n",
                   blockIdx.x, threadIdx.x, globalIdx,
                   A[globalIdx], B[globalIdx],
                   C[globalIdx]);
        }
    }
}

// ==========================================
// KERNEL 5: CUADRADO
// ==========================================

__global__ void cuadradoArreglo(
    const int* A, int* C,
    int n, bool mostrar) {

    int globalIdx =
        blockIdx.x * blockDim.x + threadIdx.x;

    if (globalIdx < n) {
        C[globalIdx] = A[globalIdx] * A[globalIdx];

        if (mostrar) {
            printf("[Equipo: 01] [DEVICE] "
                   "[Bloque CUDA: %d] [Hilo CUDA: %d] "
                   "[Indice Global: %d] "
                   "[Operacion: Cuadrado] %d * %d "
                   "-> Resultado: %d\n",
                   blockIdx.x, threadIdx.x, globalIdx,
                   A[globalIdx], A[globalIdx],
                   C[globalIdx]);
        }
    }
}

// ==========================================
// CLASE OPERACIONES ARREGLOS
// ==========================================

class OperacionesArreglos {
private:
    int n;
    int threadsPerBlock;
    bool inicializado;

    // Memoria HOST
    int* h_A;
    int* h_B;
    int* h_C;

    // Memoria DEVICE
    int* d_arreglo;
    int* d_A;
    int* d_B;
    int* d_C;

    bool revisar(cudaError_t error, const char* etapa) {
        if (error != cudaSuccess) {
            printf("ERROR en %s: %s\n",
                   etapa, cudaGetErrorString(error));
            return false;
        }
        return true;
    }

    bool verificarSuma(const int* resultado) {
        for (int i = 0; i < n; i++) {
            if (resultado[i] != h_A[i] + h_B[i])
                return false;
        }
        return true;
    }

public:
    OperacionesArreglos(int cantidad)
        : n(cantidad),
          threadsPerBlock(THREADS_PER_BLOCK),
          inicializado(false),
          h_A(nullptr), h_B(nullptr), h_C(nullptr),
          d_arreglo(nullptr), d_A(nullptr),
          d_B(nullptr), d_C(nullptr) {}

    // ======================================
    // INICIALIZAR ARREGLOS
    // ======================================

    bool inicializar() {
        if (inicializado) {
            printf("Los arreglos ya estan inicializados.\n");
            return true;
        }

        size_t bytes = (size_t)n * sizeof(int);

        h_A = new(std::nothrow) int[n];
        h_B = new(std::nothrow) int[n];
        h_C = new(std::nothrow) int[n];

        if (!h_A || !h_B || !h_C) {
            printf("Error reservando memoria HOST.\n");
            return false;
        }

        for (int i = 0; i < n; i++) {
            h_A[i] = (i % 40) + 1;
            h_B[i] = h_A[i] * 10;
            h_C[i] = 0;
        }

        if (!revisar(cudaMalloc((void**)&d_arreglo, bytes),
                     "cudaMalloc inicializacion") ||
            !revisar(cudaMalloc((void**)&d_A, bytes),
                     "cudaMalloc A") ||
            !revisar(cudaMalloc((void**)&d_B, bytes),
                     "cudaMalloc B") ||
            !revisar(cudaMalloc((void**)&d_C, bytes),
                     "cudaMalloc C"))
            return false;

        if (!revisar(cudaMemcpy(
                d_A, h_A, bytes, cudaMemcpyHostToDevice),
                "Transferencia A") ||
            !revisar(cudaMemcpy(
                d_B, h_B, bytes, cudaMemcpyHostToDevice),
                "Transferencia B"))
            return false;

        int bloques =
            (n + threadsPerBlock - 1) / threadsPerBlock;

        inicializarArreglo<<<bloques, threadsPerBlock>>>(
            d_arreglo, n, DETALLADO);

        if (!revisar(cudaGetLastError(), "Kernel inicializar") ||
            !revisar(cudaDeviceSynchronize(), "Sincronizacion"))
            return false;

        inicializado = true;

        if (DETALLADO)
            printf("Arreglos inicializados correctamente.\n");

        return true;
    }

    // ======================================
    // OPERACIONES 2 - 5
    // ======================================

    bool ejecutarOperacion(int operacion) {
        if (!inicializado) {
            printf("Primero selecciona la opcion 1.\n");
            return false;
        }

        int bloques =
            (n + threadsPerBlock - 1) / threadsPerBlock;

        const char* nombre = nullptr;

        switch (operacion) {
            case 2:
                nombre = "Suma";
                sumarArreglos<<<bloques, threadsPerBlock>>>(
                    d_A, d_B, d_C, n, DETALLADO);
                break;

            case 3:
                nombre = "Resta";
                restarArreglos<<<bloques, threadsPerBlock>>>(
                    d_A, d_B, d_C, n, DETALLADO);
                break;

            case 4:
                nombre = "Multiplicacion";
                multiplicarArreglos<<<bloques, threadsPerBlock>>>(
                    d_A, d_B, d_C, n, DETALLADO);
                break;

            case 5:
                nombre = "Cuadrado";
                cuadradoArreglo<<<bloques, threadsPerBlock>>>(
                    d_A, d_C, n, DETALLADO);
                break;

            default:
                return false;
        }

        if (!revisar(cudaGetLastError(), "Kernel operacion") ||
            !revisar(cudaDeviceSynchronize(), "Sincronizacion"))
            return false;

        size_t bytes = (size_t)n * sizeof(int);

        if (!revisar(cudaMemcpy(
                h_C, d_C, bytes, cudaMemcpyDeviceToHost),
                "Resultado DEVICE a HOST"))
            return false;

        bool correcto = true;

        for (int i = 0; i < n; i++) {
            int esperado = 0;

            if (operacion == 2)
                esperado = h_A[i] + h_B[i];
            else if (operacion == 3)
                esperado = h_A[i] - h_B[i];
            else if (operacion == 4)
                esperado = h_A[i] * h_B[i];
            else
                esperado = h_A[i] * h_A[i];

            if (h_C[i] != esperado)
                correcto = false;

            if (DETALLADO) {
                printf("[Equipo: 01] [HOST] "
                       "[Bloque CUDA: %d] [Hilo CUDA: %d] "
                       "[Indice Global: %d] "
                       "[Operacion: Verificar %s] "
                       "-> Resultado: %d\n",
                       i / threadsPerBlock,
                       i % threadsPerBlock, i,
                       nombre, h_C[i]);
            }
        }

        printf("Verificacion %s: %s\n",
               nombre, correcto ? "CORRECTA" : "ERROR");

        return correcto;
    }

    // ======================================
    // COMPARAR MEMORIAS CUDA
    // ======================================

    bool compararMemorias() {
        if (!inicializado) {
            printf("Primero inicializa los arreglos.\n");
            return false;
        }

        size_t bytes = (size_t)n * sizeof(int);
        int bloques =
            (n + threadsPerBlock - 1) / threadsPerBlock;

        int* m_A = nullptr;
        int* m_B = nullptr;
        int* m_C = nullptr;

        cudaEvent_t inicioEvento = nullptr;
        cudaEvent_t finEvento = nullptr;

        auto limpiar = [&]() {
            if (inicioEvento)
                cudaEventDestroy(inicioEvento);
            if (finEvento)
                cudaEventDestroy(finEvento);
            if (m_A)
                cudaFree(m_A);
            if (m_B)
                cudaFree(m_B);
            if (m_C)
                cudaFree(m_C);
        };

        if (!revisar(cudaMallocManaged(
                (void**)&m_A, bytes), "Managed A") ||
            !revisar(cudaMallocManaged(
                (void**)&m_B, bytes), "Managed B") ||
            !revisar(cudaMallocManaged(
                (void**)&m_C, bytes), "Managed C")) {
            limpiar();
            return false;
        }

        // Preparar memoria unificada en HOST
        for (int i = 0; i < n; i++) {
            m_A[i] = h_A[i];
            m_B[i] = h_B[i];
            m_C[i] = 0;
        }

        if (!revisar(cudaEventCreate(&inicioEvento),
                     "Crear evento inicio") ||
            !revisar(cudaEventCreate(&finEvento),
                     "Crear evento fin")) {
            limpiar();
            return false;
        }

        printf("\n=====================================\n");
        printf("EQUIPO 01 - PRUEBA MASIVA CUDA\n");
        printf("=====================================\n");
        printf("Elementos: %d\n", n);
        printf("Hilos por bloque: %d\n", threadsPerBlock);
        printf("Bloques CUDA: %d\n", bloques);
        printf("Impresiones detalladas: DESACTIVADAS\n");

        // ==================================
        // MEMORIA EXPLICITA
        // ==================================

        auto totalInicio = Reloj::now();
        auto h2dInicio = Reloj::now();

        if (!revisar(cudaMemcpy(
                d_A, h_A, bytes, cudaMemcpyHostToDevice),
                "H2D A") ||
            !revisar(cudaMemcpy(
                d_B, h_B, bytes, cudaMemcpyHostToDevice),
                "H2D B")) {
            limpiar();
            return false;
        }

        auto h2dFin = Reloj::now();

        if (!revisar(cudaEventRecord(inicioEvento),
                     "Evento inicio explicita")) {
            limpiar();
            return false;
        }

        sumarArreglos<<<bloques, threadsPerBlock>>>(
            d_A, d_B, d_C, n, false);

        if (!revisar(cudaGetLastError(),
                     "Kernel explicita") ||
            !revisar(cudaEventRecord(finEvento),
                     "Evento fin explicita") ||
            !revisar(cudaEventSynchronize(finEvento),
                     "Sincronizacion explicita")) {
            limpiar();
            return false;
        }

        float kernelExplicito = 0;

        if (!revisar(cudaEventElapsedTime(
                &kernelExplicito,
                inicioEvento, finEvento),
                "Tiempo kernel explicita")) {
            limpiar();
            return false;
        }

        auto d2hInicio = Reloj::now();

        if (!revisar(cudaMemcpy(
                h_C, d_C, bytes, cudaMemcpyDeviceToHost),
                "D2H resultado")) {
            limpiar();
            return false;
        }

        auto d2hFin = Reloj::now();

        bool validaExplicita = verificarSuma(h_C);

        auto totalFin = Reloj::now();

        double tiempoH2D =
            milisegundos(h2dInicio, h2dFin);
        double tiempoD2H =
            milisegundos(d2hInicio, d2hFin);
        double totalExplicito =
            milisegundos(totalInicio, totalFin);

        // ==================================
        // MEMORIA UNIFICADA
        // ==================================

        auto managedInicio = Reloj::now();

        if (!revisar(cudaEventRecord(inicioEvento),
                     "Evento inicio unificada")) {
            limpiar();
            return false;
        }

        sumarArreglos<<<bloques, threadsPerBlock>>>(
            m_A, m_B, m_C, n, false);

        if (!revisar(cudaGetLastError(),
                     "Kernel unificada") ||
            !revisar(cudaEventRecord(finEvento),
                     "Evento fin unificada") ||
            !revisar(cudaEventSynchronize(finEvento),
                     "Sincronizacion unificada")) {
            limpiar();
            return false;
        }

        float kernelUnificado = 0;

        if (!revisar(cudaEventElapsedTime(
                &kernelUnificado,
                inicioEvento, finEvento),
                "Tiempo kernel unificada")) {
            limpiar();
            return false;
        }

        auto lecturaInicio = Reloj::now();

        bool validaUnificada = verificarSuma(m_C);

        auto lecturaFin = Reloj::now();
        auto managedFin = Reloj::now();

        double tiempoLectura =
            milisegundos(lecturaInicio, lecturaFin);
        double totalUnificado =
            milisegundos(managedInicio, managedFin);

        // ==================================
        // RESULTADOS DE LA COMPARACION
        // ==================================

        printf("\n--- MEMORIA EXPLICITA ---\n");
        printf("Host -> Device: %.6f ms\n", tiempoH2D);
        printf("Kernel GPU:     %.6f ms\n", kernelExplicito);
        printf("Device -> Host: %.6f ms\n", tiempoD2H);
        printf("Tiempo total:   %.6f ms\n", totalExplicito);
        printf("Verificacion explicita: %s\n",
               validaExplicita ? "CORRECTA" : "ERROR");

        printf("\n--- MEMORIA UNIFICADA ---\n");
        printf("Asignacion: cudaMallocManaged\n");
        printf("Kernel GPU:      %.6f ms\n", kernelUnificado);
        printf("Lectura CPU:     %.6f ms\n", tiempoLectura);
        printf("Tiempo total:    %.6f ms\n", totalUnificado);
        printf("Verificacion unificada: %s\n",
               validaUnificada ? "CORRECTA" : "ERROR");

        printf("\n--- COMPARACION ---\n");
        printf("Total explicita: %.6f ms\n", totalExplicito);
        printf("Total unificada: %.6f ms\n", totalUnificado);

        if (totalExplicito < totalUnificado)
            printf("Menor tiempo: MEMORIA EXPLICITA\n");
        else if (totalUnificado < totalExplicito)
            printf("Menor tiempo: MEMORIA UNIFICADA\n");
        else
            printf("Tiempos iguales.\n");

        printf("Nota: los totales excluyen asignacion "
               "de memoria e inicializacion previa.\n");

        limpiar();

        return validaExplicita && validaUnificada;
    }

    // ======================================
    // DESTRUCTOR
    // ======================================

    ~OperacionesArreglos() {
        if (d_arreglo) cudaFree(d_arreglo);
        if (d_A) cudaFree(d_A);
        if (d_B) cudaFree(d_B);
        if (d_C) cudaFree(d_C);

        delete[] h_A;
        delete[] h_B;
        delete[] h_C;
    }
};

// ==========================================
// PROGRAMA PRINCIPAL
// ==========================================

int main() {
    printf("EQUIPO 01- Guzmán López Paola Nicole, Mondragón Gambino Juan Sinuhe y Morales Limón Jennifer Nataly- INICIO DE PRACTICA CUDA\n");

    cudaDeviceProp prop;

    cudaError_t error = cudaGetDeviceProperties(&prop, 0);

    if (error != cudaSuccess) {
        printf("Error GPU: %s\n",
               cudaGetErrorString(error));
        return 1;
    }

    printf("\n=== CARACTERISTICAS GPU ===\n");
    printf("GPU: %s\n", prop.name);
    printf("Multiprocesadores: %d\n",
           prop.multiProcessorCount);
    printf("Maximo hilos por bloque: %d\n",
           prop.maxThreadsPerBlock);
    printf("Memoria global: %.2f GiB\n",
           (double)prop.totalGlobalMem /
           (1024.0 * 1024 * 1024));

    OperacionesArreglos operaciones(N);

    // PRUEBA MASIVA FUERA DEL MENU
    if (MODO_MASIVO) {
        if (!operaciones.inicializar())
            return 1;

        if (!operaciones.compararMemorias())
            return 1;

        printf("\nEQUIPO 01 - FIN DE PRACTICA CUDA\n");
        return 0;
    }

    // ======================================
    // MENU INTERACTIVO DE 40 ELEMENTOS
    // ======================================

    int opcion = 0;

    do {
        printf("\n=====================================\n");
        printf("       EQUIPO 01 - MENU CUDA\n");
        printf("=====================================\n");
        printf("1. Inicializar / Crear arreglos\n");
        printf("2. Sumar arreglos\n");
        printf("3. Restar arreglos\n");
        printf("4. Multiplicar arreglos\n");
        printf("5. Calcular cuadrado de un arreglo\n");
        printf("6. Comparar modelos de memoria\n");
        printf("7. Salir\n");
        printf("=====================================\n");
        printf("Selecciona una opcion: ");
        fflush(stdout);

        if (scanf("%d", &opcion) != 1)
            break;

        switch (opcion) {
            case 1:
                operaciones.inicializar();
                break;

            case 2:
            case 3:
            case 4:
            case 5:
                operaciones.ejecutarOperacion(opcion);
                break;

            case 6:
                operaciones.compararMemorias();
                break;

            case 7:
                printf("Saliendo del programa...\n");
                break;

            default:
                printf("Opcion no valida.\n");
        }

    } while (opcion != 7);

    printf("\nEQUIPO 01 - FIN DE PRACTICA CUDA\n");

    return 0;
}

Overwriting operaciones_arreglos.cu


In [31]:
!./cuda_video_40

EQUIPO 01- Guzmán López Paola Nicole, Mondragón Gambino Juan Sinuhe y Morales Limón Jennifer Nataly- INICIO DE PRACTICA CUDA

=== CARACTERISTICAS GPU ===
GPU: Tesla T4
Multiprocesadores: 40
Maximo hilos por bloque: 1024
Memoria global: 14.56 GiB

       EQUIPO 01 - MENU CUDA
1. Inicializar / Crear arreglos
2. Sumar arreglos
3. Restar arreglos
4. Multiplicar arreglos
5. Calcular cuadrado de un arreglo
6. Comparar modelos de memoria
7. Salir
Selecciona una opcion: 1
[Equipo: 01] [DEVICE] [Bloque CUDA: 1] [Hilo CUDA: 0] [Indice Global: 10] [Operacion: Inicializar] -> Valor: 11
[Equipo: 01] [DEVICE] [Bloque CUDA: 1] [Hilo CUDA: 1] [Indice Global: 11] [Operacion: Inicializar] -> Valor: 12
[Equipo: 01] [DEVICE] [Bloque CUDA: 1] [Hilo CUDA: 2] [Indice Global: 12] [Operacion: Inicializar] -> Valor: 13
[Equipo: 01] [DEVICE] [Bloque CUDA: 1] [Hilo CUDA: 3] [Indice Global: 13] [Operacion: Inicializar] -> Valor: 14
[Equipo: 01] [DEVICE] [Bloque CUDA: 1] [Hilo CUDA: 4] [Indice Global: 14] [Operacio

In [29]:
!printf '1\n6\n7\n' | ./operaciones_arreglos



EQUIPO 01 - INICIO DE PRACTICA CUDA

=== CARACTERISTICAS GPU ===
GPU: Tesla T4
Multiprocesadores: 40
Maximo hilos por bloque: 1024
Memoria global: 14.56 GiB

       EQUIPO 01 - MENU CUDA
1. Inicializar / Crear arreglos
2. Sumar arreglos
3. Restar arreglos
4. Multiplicar arreglos
5. Calcular cuadrado de un arreglo
6. Comparar modelos de memoria
7. Salir
Selecciona una opcion: [Equipo: 01] [DEVICE] [Bloque CUDA: 1] [Hilo CUDA: 0] [Indice Global: 10] [Operacion: Inicializar] -> Valor: 11
[Equipo: 01] [DEVICE] [Bloque CUDA: 1] [Hilo CUDA: 1] [Indice Global: 11] [Operacion: Inicializar] -> Valor: 12
[Equipo: 01] [DEVICE] [Bloque CUDA: 1] [Hilo CUDA: 2] [Indice Global: 12] [Operacion: Inicializar] -> Valor: 13
[Equipo: 01] [DEVICE] [Bloque CUDA: 1] [Hilo CUDA: 3] [Indice Global: 13] [Operacion: Inicializar] -> Valor: 14
[Equipo: 01] [DEVICE] [Bloque CUDA: 1] [Hilo CUDA: 4] [Indice Global: 14] [Operacion: Inicializar] -> Valor: 15
[Equipo: 01] [DEVICE] [Bloque CUDA: 1] [Hilo CUDA: 5] [Indice 

In [28]:
!TZ=America/Mexico_City date "+Fecha: %d/%m/%Y | Hora: %H:%M:%S"; ./cuda_video_masivo



Fecha: 04/11/2026 | Hora: 01:54:06
EQUIPO 01- Guzmán López Paola Nicole, Mondragón Gambino Juan Sinuhe y Morales Limón Jennifer Nataly- INICIO DE PRACTICA CUDA

=== CARACTERISTICAS GPU ===
GPU: Tesla T4
Multiprocesadores: 40
Maximo hilos por bloque: 1024
Memoria global: 14.56 GiB

EQUIPO 01 - PRUEBA MASIVA CUDA
Elementos: 4000000
Hilos por bloque: 256
Bloques CUDA: 15625
Impresiones detalladas: DESACTIVADAS

--- MEMORIA EXPLICITA ---
Host -> Device: 6.862699 ms
Kernel GPU:     0.206688 ms
Device -> Host: 3.669379 ms
Tiempo total:   14.992670 ms
Verificacion explicita: CORRECTA

--- MEMORIA UNIFICADA ---
Asignacion: cudaMallocManaged
Kernel GPU:      12.259360 ms
Lectura CPU:     7.308833 ms
Tiempo total:    19.595138 ms
Verificacion unificada: CORRECTA

--- COMPARACION ---
Total explicita: 14.992670 ms
Total unificada: 19.595138 ms
Menor tiempo: MEMORIA EXPLICITA
Nota: los totales excluyen asignacion de memoria e inicializacion previa.

EQUIPO 01 - FIN DE PRACTICA CUDA


In [27]:
!nvcc -O2 -std=c++17 operaciones_arreglos.cu -o cuda_video_masivo && echo "COMPILACION MASIVA CORRECTA"


operaciones_arreglos.cu(26): warning #177-D: variable "HILOS" was declared but never referenced
  constexpr int HILOS = 256;
                ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

COMPILACION MASIVA CORRECTA


In [22]:

!TZ=America/Mexico_City date "+Fecha: %d/%m/%Y | Hora: %H:%M:%S"; printf '1\n2\n3\n4\n5\n6\n7\n' | ./cuda_video_40


Fecha: 04/11/2026 | Hora: 01:45:17
EQUIPO 01- Guzmán López Paola Nicole, Mondragón Gambino Juan Sinuhe y Morales Limón Jennifer Nataly- INICIO DE PRACTICA CUDA

=== CARACTERISTICAS GPU ===
GPU: Tesla T4
Multiprocesadores: 40
Maximo hilos por bloque: 1024
Memoria global: 14.56 GiB

       EQUIPO 01 - MENU CUDA
1. Inicializar / Crear arreglos
2. Sumar arreglos
3. Restar arreglos
4. Multiplicar arreglos
5. Calcular cuadrado de un arreglo
6. Comparar modelos de memoria
7. Salir
Selecciona una opcion: [Equipo: 01] [DEVICE] [Bloque CUDA: 1] [Hilo CUDA: 0] [Indice Global: 10] [Operacion: Inicializar] -> Valor: 11
[Equipo: 01] [DEVICE] [Bloque CUDA: 1] [Hilo CUDA: 1] [Indice Global: 11] [Operacion: Inicializar] -> Valor: 12
[Equipo: 01] [DEVICE] [Bloque CUDA: 1] [Hilo CUDA: 2] [Indice Global: 12] [Operacion: Inicializar] -> Valor: 13
[Equipo: 01] [DEVICE] [Bloque CUDA: 1] [Hilo CUDA: 3] [Indice Global: 13] [Operacion: Inicializar] -> Valor: 14
[Equipo: 01] [DEVICE] [Bloque CUDA: 1] [Hilo CUDA:

In [19]:

!nvcc -O2 -std=c++17 operaciones_arreglos.cu -o cuda_video_40


operaciones_arreglos.cu(21): warning #177-D: variable "HILOS" was declared but never referenced
  constexpr int HILOS = 10;
                ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

